## 🏗️ Mini Project

#### "Customer Lifetime Value (CLV) Predictor": Gradient Boosting vs Random Forest

This project reuses the CLV dataset from the Random Forest mini-project. Tasks:

1. Train a Gradient Boosting model
2. Run a combined grid search over n_estimators and learning_rate (as in the Random Forest project) and find the best combination
3. Compare accuracy with Random Forest: which performs better on this dataset?
4. Compare training time too (using the time module): is Gradient Boosting really slower?

In [1]:
import numpy as np
import pandas as pd

np.random.seed(21)
n_customers = 400

# Relevant features
avg_order_value = np.random.uniform(20, 200, n_customers)
purchase_frequency = np.random.poisson(5, n_customers)  # per year
tenure_months = np.random.uniform(1, 60, n_customers)
support_tickets = np.random.poisson(2, n_customers)

# Irrelevant/noise features (real-world data always has some!)
random_noise_1 = np.random.randn(n_customers)
random_noise_2 = np.random.randn(n_customers)
signup_day_of_week = np.random.randint(0, 7, n_customers)

X = np.column_stack([avg_order_value, purchase_frequency, tenure_months, 
                       support_tickets, random_noise_1, random_noise_2, signup_day_of_week])

# True relationship (complex, non-linear interactions)
CLV = (avg_order_value * purchase_frequency * 0.5 
       + tenure_months * 8 
       - support_tickets * 15 
       + np.random.normal(0, 50, n_customers))
CLV = np.maximum(CLV, 0)  # CLV can't be negative

y = CLV

In [2]:
import time
from sklearn. model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn. ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error

gb_results = []
rf_results = []

for seed in [22, 32, 42, 52]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= seed)        

    for estimators in [50, 100, 150, 200]:
        for depth in [1,2,3,4]:
            for rate in [0.01, 0.05, 0.1, 0.5]:
                gb_model = GradientBoostingRegressor(
                    n_estimators= estimators,
                    max_depth=depth,
                    learning_rate= rate,
                    random_state= seed
                )
                start_train = time.time()
                gb_model.fit(X_train, y_train)
                end_train = time.time()
                train_time = end_train - start_train

                start_pred = time.time()
                y_pred = gb_model.predict(X_test)
                end_pred = time.time()
                pred_time = end_pred - start_pred

                mse_error = mean_squared_error(y_test, y_pred)

                gb_results.append(

                    {'Seed': seed,
                     'Estimators': estimators,
                     'Learning Rate': rate,
                     'Depth': depth,
                     'MSE': round(mse_error,2),
                     'Train Time (sec)': round(train_time, 2),
                     'Predict Time (sec)': round(pred_time, 2)
                    }
                )
            for feature in [2, 4, 7]:
                rf = RandomForestRegressor(
                      n_estimators= estimators, 
                      max_depth= depth, 
                      max_features= feature,
                        random_state= seed)

                start_train = time.time()
                rf.fit(X_train, y_train)
                end_train = time.time()
                train_time = end_train - start_train

                start_pred = time.time()
                rf_pred = rf.predict(X_test)
                end_pred = time.time()
                pred_time = end_pred - start_pred

                rf_mse = mean_squared_error(y_test, rf_pred)
                
                rf_results.append(
                    {
                        "Seed": seed,
                        "Estimators": estimators,
                        "Max Features": feature,
                        "Max Depth": depth,
                        "MSE": round(rf_mse, 2),
                        'Train Time (sec)': round(train_time, 2),
                        'Predict Time (sec)': round(pred_time, 2)
                    }
                )
                
                

gb = pd.DataFrame(gb_results)    
df_rf = pd.DataFrame(rf_results)    

In [3]:
top5_gb = gb.sort_values(by = "MSE", ascending= True).head(5)
top5_gb

,Seed,Estimators,Learning Rate,Depth,MSE,Train Time (sec),Predict Time (sec)
41,22,150,0.05,3,3646.57,0.17,0.00
42,22,150,0.10,3,3676.87,0.15,0.00
57,22,200,0.05,3,3711.30,5.24,0.03
26,22,100,0.10,3,3724.95,0.08,0.00
166,42,150,0.10,2,3751.31,0.21,0.00


In [4]:
top5_rf = df_rf.sort_values(by = "MSE", ascending= True).head(5)
top5_rf

,Seed,Estimators,Max Features,Max Depth,MSE,Train Time (sec),Predict Time (sec)
167,52,100,7,4,6644.72,0.12,0.00
155,52,50,7,4,6652.93,0.03,0.00
179,52,150,7,4,6807.64,0.18,0.00
191,52,200,7,4,6839.01,0.24,0.00
71,32,100,7,4,7753.97,4.30,0.05
